# Evaluation metrics and inference: deterministic example

Count completed synthetic positions, measure drawdown, and keep path and paired-observation units visible.

In [1]:
from math import isclose

import numpy as np

from evaluation.metrics.bootstrap import paired_metric_difference_ci
from evaluation.metrics.performance import completed_position_trade_pnls, max_drawdown

legs = [
    {"position_id": "A", "type": "tp_partial_close", "net_pnl": 4, "position_complete": False},
    {"position_id": "A", "type": "close", "net_pnl": 6, "position_complete": True},
    {"position_id": "B", "type": "tp_partial_close", "net_pnl": -2, "position_complete": False},
]
pnls, realized_fills, open_positions = completed_position_trade_pnls(legs)
assert (pnls, realized_fills, open_positions) == ([10], 3, 1)

equity = [100, 120, 90, 110]
drawdown = max_drawdown(equity)
payout = 5
economic_return = (equity[-1] + payout - equity[0]) / equity[0]
assert isclose(drawdown, 0.25)
assert isclose(economic_return, 0.15)

treatment = np.array([0.03, 0.00, 0.04, 0.01])
control = np.array([0.02, -0.01, 0.03, 0.00])
interval = paired_metric_difference_ci(
    treatment, control, np.mean, n_boot=32, rng_seed=42, block_len=2
)
assert all(isclose(value, 0.01) for value in interval.as_tuple())

print(
    f"Synthetic counts: {len(pnls)} completed, {realized_fills} realized fills, {open_positions} open"
)
print(f"Synthetic drawdown: {drawdown:.0%}, payout-aware return: {economic_return:.0%}")
print(f"Synthetic paired return difference: {interval.estimate:.2%}")

Synthetic counts: 1 completed, 3 realized fills, 1 open
Synthetic drawdown: 25%, payout-aware return: 15%
Synthetic paired return difference: 1.00%


The constant difference makes the simulated paired interval collapse to its point estimate. The fixture uses four synthetic observations and supplies no between-seed or market evidence.